# 🎀 林星蕾 LoRA 訓練營 — Kaggle 版本
## 設備：T4 GPU · SDXL · 1024×1024 · 118張圖片


In [ ]:
# === 1. 安裝依賴 ===
!pip install -q --upgrade pip
!pip install -q accelerate transformers ftfy tensorboard safetensors lion-pytorch huggingface_hub opencv-python-headless
print("✅ 依賴安裝完成")

In [ ]:
# === 2. 設定 Accelerate ===
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision="bf16")
print("✅ Accelerate 設定完成 (bf16)")

In [ ]:
# === 3. 下載訓練素材 ===
import urllib.request, os

ZIP_URL = "https://github.com/vito0527opencode/lora-training/raw/main/lora_training_data.zip"
ZIP_PATH = "/kaggle/working/lora_training_data.zip"

if not os.path.exists(ZIP_PATH):
    print("📦 自動下載訓練素材...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    print(f"✅ 下載完成: {os.path.getsize(ZIP_PATH)//1024//1024} MB")
else:
    print("✅ zip 已存在")

assert os.path.exists(ZIP_PATH), "zip 下載失敗！"
print(f"📦 zip 大小: {os.path.getsize(ZIP_PATH)//1024//1024} MB")

In [ ]:
# === 4. 解壓縮 + 產生 caption 檔案 ===
import zipfile, glob, os, random

zips = glob.glob("/kaggle/working/*.zip")
if zips:
    with zipfile.ZipFile(zips[0], "r") as z:
        z.extractall("/kaggle/working/training_data")
    print("✅ 解壓縮完成")

caption_candidates = [
    "/kaggle/working/training_data/captions.txt",
    "/kaggle/working/training_data/processed/captions.txt"
]
caption_file = None
for c in caption_candidates:
    if os.path.exists(c):
        caption_file = c
        break

assert caption_file, f"找不到 captions.txt！檢查過: {caption_candidates}"
print(f"✅ Caption 檔: {caption_file}")

processed_dir = os.path.dirname(caption_file)
print(f"✅ 訓練資料目錄: {processed_dir}")

TRIGGER = "linxinglei"
caption_map = {}
with open(caption_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if "|" not in line:
            continue
        img_name, cap = line.split("|", 1)
        caption_map[img_name.strip()] = cap.strip()

png_files = [f for f in os.listdir(processed_dir) if f.endswith(".png")]
txt_count = 0
for png in png_files:
    txt_path = os.path.join(processed_dir, png.rsplit(".", 1)[0] + ".txt")
    cap = caption_map.get(png, f"a woman with long black hair")
    with open(txt_path, "w", encoding="utf-8") as cf:
        cf.write(f"{TRIGGER}, {cap}")
    txt_count += 1

print(f"✅ 產生 {txt_count} 個 caption 檔案")

samples = random.sample(png_files, min(5, len(png_files)))
for s in sorted(samples):
    txt_path = os.path.join(processed_dir, s.rsplit(".", 1)[0] + ".txt")
    cap = open(txt_path).read().strip() if os.path.exists(txt_path) else "(無)"
    print(f"  📸 {s} → {cap}")

In [ ]:
# === 5. 克隆 Kohya sd-scripts ===
!git clone -q https://github.com/kohya-ss/sd-scripts.git /kaggle/working/sd-scripts

TRAIN_SCRIPT = "/kaggle/working/sd-scripts/sdxl_train_network.py"
assert os.path.exists(TRAIN_SCRIPT), f"❌ 找不到腳本: {TRAIN_SCRIPT}"

print(f"✅ sd-scripts 安裝完成")
print(f"📝 訓練腳本: {TRAIN_SCRIPT}")

In [ ]:
# === 6. 🚀 開始訓練（SDXL LoRA）===
import os, subprocess, sys

processed_dir = None
for root, dirs, files in os.walk("/kaggle/working/training_data"):
    if "captions.txt" in files:
        processed_dir = root
        break

if not processed_dir:
    processed_dir = "/kaggle/working/training_data/processed"

TRAIN_SCRIPT = "/kaggle/working/sd-scripts/sdxl_train_network.py"
assert os.path.exists(TRAIN_SCRIPT), f"❌ 找不到腳本: {TRAIN_SCRIPT}"

print(f"📂 訓練資料: {processed_dir}")
print(f"📂 輸出目錄: /kaggle/working/output")

import torch
print()
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   CUDA: {torch.version.cuda}")
else:
    print("❌ 警告：無 GPU！")

os.makedirs("/kaggle/working/output", exist_ok=True)

cmd = (
    f"accelerate launch "
    f"{TRAIN_SCRIPT} "
    f"--pretrained_model_name_or_path=stabilityai/stable-diffusion-xl-base-1.0 "
    f"--train_data_dir={processed_dir} "
    f"--output_dir=/kaggle/working/output "
    f"--output_name=linxinglei_lora_v1 "
    f"--resolution=1024,1024 "
    f"--train_batch_size=1 "
    f"--network_module=networks.lora "
    f"--network_dim=32 "
    f"--network_alpha=16 "
    f"--max_train_steps=1200 "
    f"--learning_rate=1e-4 "
    f"--optimizer_type=AdamW8bit "
    f"--mixed_precision=bf16 "
    f"--save_every_n_epochs=2 "
    f"--save_model_as=safetensors "
    f"--caption_extension=.txt "
    f"--cache_latents "
    f"--lr_scheduler=cosine "
    f"--lr_warmup_steps=50 "
    f"--enable_bucket "
    f"--min_bucket_reso=256 "
    f"--max_bucket_reso=2048 "
    f"--keep_tokens=0 "
    f"--no_half_vae "
    f"--gradient_accumulation_steps=1"
)

print("🔥 開始訓練... (約 1-2 小時)")
print("="*50)

result = subprocess.run(cmd, shell=True, capture_output=False)
print()
print("="*50)
if result.returncode == 0:
    print("🎉 訓練完成！")
else:
    print("❌ 訓練失敗！")

In [ ]:
# === 7. 下載 LoRA ===
import glob, os
from datetime import datetime

print("📦 尋找 LoRA 檔案...")

safetensors_files = glob.glob("/kaggle/working/output/**/*.safetensors", recursive=True)

if safetensors_files:
    latest = max(safetensors_files, key=os.path.getmtime)
    size_mb = os.path.getsize(latest) / 1024 / 1024
    print(f"📦 找到 LoRA: {latest}")
    print(f"📏 大小: {size_mb:.1f} MB")
    print()
    print("⬇️  下載...")
    from google.colab import files
    files.download(latest)
else:
    print("⚠️ 找不到 .safetensors 檔案")
    !find /kaggle/working/output -type f 2>/dev/null | head -20

---
## 🎉 完成！

**使用方式**：
- ComfyUI / A1111 / SD.Next 載入 
- 觸發詞：
- 權重建議：0.6 ~ 0.8